In [1]:
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
from ddp import *

In [2]:
benchmark_settings = {
    "model_size": ["tiny", "small", "deep-narrow", "ffn-heavy"],
    "d_model": [384, 768, 512, 768],
    "d_ff": [1536, 3072, 2048, 6144],
    "num_layers": [6, 12, 20, 12],
    "num_heads": [6, 12, 8, 12]
}
WORLD_SIZE = 2
WARMUP_ITERS = 5
ddp_map = {
    "DDPNaive": DDPNaive,
    "DDPOverlap": DDPOverlap
}

params = ["d_model", "d_ff", "num_layers", "num_heads"]
length = len(benchmark_settings["model_size"])
for key in params:
    assert len(benchmark_settings[key]) == length, f"Length mismatch for {key}"

output_dict = {key: list() for key in list(benchmark_settings.keys()) + ["ddp_class", "forward_avg", "forward_std", "backward_avg", "backward_std", "optimizer_avg", "optimizer_std"]}
for index in range(length):
    for ddp_key, ddp_class in ddp_map.items():
        input_dict = {key: benchmark_settings[key][index] for key in params}

        ctx = mp.get_context("spawn")
        queue = ctx.Queue()
        mp.spawn(benchmark_python_worker, nprocs=WORLD_SIZE, args=(WORLD_SIZE, queue, ddp_class, input_dict), join=True)
        results = queue.get()
        
        output_dict["ddp_class"].append(ddp_key)
        for key, val in results.items():
            output_dict[f"{key}_avg"].append(np.mean(val[WARMUP_ITERS:]))
            output_dict[f"{key}_std"].append(np.std(val[WARMUP_ITERS:]))
        for u, v in benchmark_settings.items():
            output_dict[u].append(v[index])
        print(f"Done {benchmark_settings['model_size'][index]}: ", f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
        torch.cuda.reset_peak_memory_stats()

output_df = pd.DataFrame({**output_dict})
output_df

Done tiny:  peak=0.0GB
Done tiny:  peak=0.0GB
Done small:  peak=0.0GB
Done small:  peak=0.0GB
Done deep-narrow:  peak=0.0GB
Done deep-narrow:  peak=0.0GB
Done ffn-heavy:  peak=0.0GB
Done ffn-heavy:  peak=0.0GB


,model_size,d_model,d_ff,num_layers,num_heads,ddp_class,forward_avg,forward_std,backward_avg,backward_std,optimizer_avg,optimizer_std
0,tiny,384,1536,6,6,DDPNaive,13.771181,7.188945,163.801373,27.586010,17.391399,7.994021
1,tiny,384,1536,6,6,DDPOverlap,15.276835,8.170711,93.002459,23.039171,24.570601,14.427862
2,small,768,3072,12,12,DDPNaive,33.886621,11.836789,534.856269,33.443989,42.687203,12.395898
3,small,768,3072,12,12,DDPOverlap,36.909772,20.758205,385.084898,55.979136,41.832450,21.458637
4,deep-narrow,512,2048,20,8,DDPNaive,38.888194,15.651522,516.002274,33.174389,66.246233,24.599198
5,deep-narrow,512,2048,20,8,DDPOverlap,38.678207,17.625290,336.733092,56.192001,60.408904,33.658855
6,ffn-heavy,768,6144,12,12,DDPNaive,42.122336,12.350411,757.548519,41.572186,61.517839,13.414090
7,ffn-heavy,768,6144,12,12,DDPOverlap,41.292950,14.393674,545.863719,48.178798,74.207862,32.256901
